# Aim 1 — Orthogonality Spectrum Corpus Generation

Generates 500–1000 concept pairs per category (5 categories) via the SURF AI Hub (WiLLMa) OpenAI-compatible API.

**Logic lives in `.py` modules** in this folder; this notebook only configures and launches the run.

| Module | Role |
|---|---|
| `config.py` | Endpoint, model, batch size, output paths |
| `domain_seed.py` | `domain_seeds_map` (5 categories × seeds) |
| `prompts.py` | Category meta-prompt templates |
| `generate_corpus.py` | Nested loop + API + JSON parse + save |

## 0. Install dependencies (once)

```bash
pip install -r requirements.txt
```

Set your WiLLMa API key in the environment (or in the next cell):

```bash
export WILLMA_API_KEY="your-key-here"
```

Or copy `.env.example` → `.env` and fill in the key.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Resolve package location even if Jupyter cwd is the repo root or this folder
_cwd = Path.cwd().resolve()
_candidates = [
    _cwd,
    _cwd / "vocabulary_gen_aim1_AND",
    _cwd / "src" / "vocabulary_gen_aim1_AND",
    Path("/home/WUR/lu087/Github/ect/src/vocabulary_gen_aim1_AND"),
]
NOTEBOOK_DIR = next(
    (p for p in _candidates if (p / "generate_corpus.py").exists()),
    _cwd,
)
SRC_DIR = NOTEBOOK_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(NOTEBOOK_DIR / ".env")
load_dotenv()  # also pick up a repo-root .env if present

# Optional: paste key here for a local session (prefer env / .env instead)
# os.environ["WILLMA_API_KEY"] = "willma-XXXXXXXX"

import importlib
import vocabulary_gen_aim1_AND
import vocabulary_gen_aim1_AND.config as _cfg
import vocabulary_gen_aim1_AND.prompts as _prompts
import vocabulary_gen_aim1_AND.generate_corpus as _gen

importlib.reload(_cfg)
importlib.reload(_prompts)
importlib.reload(_gen)
importlib.reload(vocabulary_gen_aim1_AND)

from vocabulary_gen_aim1_AND import (
    BASE_URL,
    MODEL,
    corpus_summary,
    domain_seeds_map,
    run_generation,
)
from vocabulary_gen_aim1_AND.config import (
    DEFAULT_OUTPUT_CSV,
    DEFAULT_OUTPUT_JSONL,
    PAIRS_PER_BATCH,
    TARGET_PAIRS_PER_CATEGORY,
)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("BASE_URL:", BASE_URL)
print("MODEL:", MODEL)
print("seeds per category:", {k: len(v) for k, v in domain_seeds_map.items()})
print("API key set:", bool(os.getenv("WILLMA_API_KEY") or os.getenv("OPENAI_API_KEY")))

NOTEBOOK_DIR: /home/WUR/lu087/Github/ect/src/vocabulary_gen_aim1_AND
BASE_URL: https://api.willma.surf.nl/v0
MODEL: Qwen/Qwen3.6-35B-A3B-FP8
seeds per category: {1: 20, 2: 20, 3: 20, 4: 20, 5: 20}
API key set: True


## 1. Dry-run prompt check (no API calls)

Confirms nested loops and dynamic category constraints before spending tokens.

In [2]:
_ = run_generation(
    dry_run=True,
    categories=[1],  # sample one category
    shuffle_seeds=False,
    verbose=True,
)

base_url=https://api.willma.surf.nl/v0
model=Qwen/Qwen3.6-35B-A3B-FP8
existing_pairs=2990
target_per_category=750
structured_output=json_schema (fallback: json_object)
DRY RUN — no API calls
[cat 1] already have 799 pairs (≥ target); skipping
Done. Per-category counts: {1: 799, 2: 795, 3: 750, 4: 646, 5: 0}
Wrote JSONL → /home/WUR/lu087/Github/ect/src/vocabulary_gen_aim1_AND/output/orthogonality_spectrum_corpus.jsonl
Wrote CSV   → /home/WUR/lu087/Github/ect/src/vocabulary_gen_aim1_AND/output/orthogonality_spectrum_corpus.csv


## 2. Launch full corpus generation

- **50 pairs / API call** per domain seed
- Outer loop: categories 1–5; inner loop: seeds
- Deduplicates within category; resumes from existing JSONL if present
- Writes `output/orthogonality_spectrum_corpus.jsonl` and `.csv`

In [3]:
# Tunables for your run
CATEGORIES = [1, 2, 3, 4, 5]  # subset for a smoke test, e.g. [1]
TARGET = TARGET_PAIRS_PER_CATEGORY  # default 750 (within 500–1000)
BATCH = PAIRS_PER_BATCH  # 50

corpus = run_generation(
    # api_key="...",  # only if not set via WILLMA_API_KEY / .env
    base_url=BASE_URL,
    model=MODEL,
    categories=CATEGORIES,
    pairs_per_batch=BATCH,
    target_per_category=TARGET,
    shuffle_seeds=True,
    resume=True,
    dry_run=False,
    output_jsonl=DEFAULT_OUTPUT_JSONL,
    output_csv=DEFAULT_OUTPUT_CSV,
    verbose=True,
)

corpus_summary(corpus)

base_url=https://api.willma.surf.nl/v0
model=Qwen/Qwen3.6-35B-A3B-FP8
existing_pairs=2990
target_per_category=750
structured_output=json_schema (fallback: json_object)
[cat 1] already have 799 pairs (≥ target); skipping
[cat 2] already have 795 pairs (≥ target); skipping
[cat 3] already have 750 pairs (≥ target); skipping
[cat 4] seed='Childhood Memories Materializing as Sentient Vehicles' (have 646)
  SKIP after retries: Empty model response
[cat 4] seed='Historical Figures from Antiquity placed in Hard Sci-Fi Scenarios' (have 646)
  +50 new (parsed 50; cat total 696)
[cat 4] seed='Philosophical Ideologies Manifesting as Edible Desserts' (have 696)
  +49 new (parsed 50; cat total 745)
[cat 4] seed='Planetary Bodies and Galaxies performing Mundane Household Chores' (have 745)
  +50 new (parsed 50; cat total 795)
[cat 5] seed='Absolute Certainty combined with Radical Uncertainty' (have 0)
  +50 new (parsed 50; cat total 50)
[cat 5] seed='Biological Life and Vitality combined with Absolu

{'total': 3885,
 'per_category': {'Redundant / Strict Entailment': 799,
  'Typical Composition': 795,
  'Uncommon / Orthogonal': 750,
  'Surreal / Out-of-Distribution': 795,
  'Mutually Exclusive / Paradoxical': 746},
 'output_dir': '/home/WUR/lu087/Github/ect/src/vocabulary_gen_aim1_AND/output'}

## 3. Inspect results

In [ ]:
import pandas as pd

df = pd.read_csv(DEFAULT_OUTPUT_CSV)
display(df.groupby(["category_id", "category_name"]).size().rename("n_pairs").reset_index())
df.sample(min(10, len(df)), random_state=0) if len(df) else df